# Лабораторная работа №3. Градиентный спуск

**Цель:** реализовать с нуля градиентный (GD) и стохастический градиентный спуск (SGD) для линейной регрессии, исследовать влияние скорости обучения, сравнить скорость сходимости методов и изучить L1/L2-регуляризацию.

**План:**
1. Генерация и подготовка данных
2. Градиентный спуск (собственная реализация)
3. Эксперименты со скоростью обучения
4. Стохастический градиентный спуск
5. Сравнение GD и SGD
6. Регуляризация (L1 / L2) и зависимость весов от коэффициента регуляризации

In [ ]:
!pip install numpy pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

# ВАШ НОМЕР ВАРИАНТА (чётный -> L1, нечётный -> L2). Измените при необходимости.
VARIANT = 1
REG_TYPE = 'l1' if VARIANT % 2 == 0 else 'l2'
print('Тип регуляризации для варианта', VARIANT, '->', REG_TYPE.upper())

## 1. Генерация и подготовка данных

Генерируем датасет через `make_regression`: 8 признаков (из них 4 информативных), то есть вместе со свободным членом 9 весов (требуется не менее 3). Неинформативные признаки нужны, чтобы L1-регуляризация имела что занулять.

Признаки масштабируем (`StandardScaler`, параметры считаются по обучающей выборке), к матрице добавляем столбец единиц для свободного члена $w_0$.

In [ ]:
n_features = 8
X_raw, y, true_coef = make_regression(
    n_samples=1000, n_features=n_features, n_informative=4,
    noise=10.0, bias=5.0, coef=True, random_state=42
)

print('Форма X:', X_raw.shape, ' форма y:', y.shape)
print('Истинные коэффициенты:', np.round(true_coef, 2))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.25, random_state=42
)

scaler = StandardScaler().fit(X_train_raw)
def add_bias(A):
    return np.hstack([np.ones((A.shape[0], 1)), A])

X_train = add_bias(scaler.transform(X_train_raw))
X_test  = add_bias(scaler.transform(X_test_raw))
print('Размер обучающей матрицы (с bias):', X_train.shape)

## 2. Градиентный спуск

Функция потерь — MSE:
$$Q(w, X) = \frac{1}{l}\|Xw - y\|^2, \qquad \nabla Q(w) = \frac{2}{l} X^T (Xw - y).$$

Алгоритм:
1. инициализировать веса (нули) и $\varepsilon$;
2. цикл по числу итераций;
3. $w^k = w^{k-1} - \eta \nabla Q(w^{k-1}, X)$;
4. выйти из цикла, если $\|w^k - w^{k-1}\| < \varepsilon$.

Функция сразу написана с параметрами для последующих пунктов (`stochastic` — п. 4, `reg`/`alpha` — п. 6); при значениях по умолчанию это обычный (пакетный) градиентный спуск.

In [ ]:
def mserror(X, w, y):
    return np.mean((X @ w - y) ** 2)


def gradient_descent(X, y, eta=0.1, max_iter=10_000, eps=1e-8,
                     stochastic=False, reg=None, alpha=0.0,
                     w_init=None, seed=None, verbose=False):
    """
    Градиентный спуск для линейной регрессии (MSE).

    stochastic=False -> на каждой итерации градиент считается по всей выборке (GD)
    stochastic=True  -> на каждой итерации выбирается один случайный объект (SGD)
    reg=None | 'l2' | 'l1', alpha - коэффициент регуляризации (свободный член w[0] не штрафуется).

    Возвращает: итоговые веса, список MSE на ВСЕЙ выборке по итерациям, историю весов.
    """
    rng = np.random.default_rng(seed)
    l, n = X.shape
    w = np.zeros(n) if w_init is None else w_init.copy()
    errors, w_hist = [], [w.copy()]

    for k in range(max_iter):
        np.seterr(over='ignore', invalid='ignore')
        if stochastic:
            i = rng.integers(l)                 # случайный объект
            Xb, yb = X[i:i + 1], y[i:i + 1]
        else:
            Xb, yb = X, y

        grad = 2 / len(yb) * Xb.T @ (Xb @ w - yb)

        if reg == 'l2':                          # Q + alpha*||w||^2
            pen = 2 * alpha * w
            pen[0] = 0
            grad = grad + pen

        new_w = w - eta * grad

        if reg == 'l1':                          # Q + alpha*||w||_1 (проксимальный шаг)
            thr = eta * alpha
            shrunk = np.sign(new_w) * np.maximum(np.abs(new_w) - thr, 0)
            shrunk[0] = new_w[0]
            new_w = shrunk

        if not np.all(np.isfinite(new_w)):       # расходимость
            errors.append(np.inf)
            if verbose:
                print(f'Iter {k}: расходимость')
            break

        dist = np.linalg.norm(new_w - w)
        w = new_w
        errors.append(mserror(X, w, y))
        w_hist.append(w.copy())

        if verbose and k % 100 == 0:
            print(f'Iter {k}: error = {errors[-1]:.4f}')

        if dist < eps:                           # критерий останова
            break

    return w, np.array(errors), np.array(w_hist)

In [ ]:
w_gd, err_gd, hist_gd = gradient_descent(X_train, y_train, eta=0.1, max_iter=10_000, eps=1e-8)

# Эталоны: нормальное уравнение и sklearn
w_normal = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y_train
lr = LinearRegression(fit_intercept=False).fit(X_train, y_train)

print('Итераций GD:', len(err_gd))
print(pd.DataFrame({'GD': w_gd, 'норм. уравнение': w_normal, 'sklearn': lr.coef_},
                   index=[f'w{i}' for i in range(len(w_gd))]).round(4))
print('\nMSE train (GD):', round(mserror(X_train, w_gd, y_train), 4),
      '| MSE test (GD):', round(mserror(X_test, w_gd, y_test), 4))

Полученные веса совпадают с аналитическим решением (метод наименьших квадратов) и с `sklearn`, значит реализация корректна.

## 3. Эксперименты со скоростью обучения

Запускаем GD с разными значениями $\eta$ при одинаковых прочих параметрах.

In [ ]:
lam_max = np.linalg.eigvalsh(2 / len(y_train) * X_train.T @ X_train).max()
print(f'Наибольшее собственное число гессиана: {lam_max:.3f} -> граница устойчивости eta < 2/lambda_max = {2 / lam_max:.3f}')
etas = [0.001, 0.01, 0.1, 0.5, 0.7, 0.9, 1.05]
rows = []
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

for eta in etas:
    w_e, err_e, _ = gradient_descent(X_train, y_train, eta=eta, max_iter=2000, eps=1e-8)
    diverged = not np.isfinite(err_e[-1]) or err_e[-1] > 1e6
    rows.append({
        'eta': eta,
        'итераций': len(err_e),
        'финальная MSE': err_e[-1],
        'расстояние до МНК': np.linalg.norm(w_e - w_normal),
        'расходится': diverged
    })
    if not diverged:
        ax[0].plot(err_e, label=f'η={eta}')
    else:
        ax[1].plot(np.where(np.isfinite(err_e), err_e, np.nan)[:60], label=f'η={eta}')

ax[0].set_yscale('log'); ax[0].set_xscale('log')
ax[0].set_title('Сходящиеся значения η'); ax[0].set_xlabel('итерация'); ax[0].set_ylabel('MSE'); ax[0].legend()
ax[1].set_yscale('log'); ax[1].set_title('Расходящееся η'); ax[1].set_xlabel('итерация'); ax[1].legend()
plt.tight_layout(); plt.show()

pd.DataFrame(rows)

### Выводы по скорости обучения

- **Слишком маленькое $\eta$ (0.001)** — спуск очень медленный: за отведённые 2000 итераций алгоритм не доходит до минимума (расстояние до решения МНК заметно отличается от нуля). Кроме того, при малом $\eta$ шаги малы сами по себе, и критерий $\|w^k-w^{k-1}\|<\varepsilon$ может сработать до реальной сходимости.
- **Умеренное $\eta$ (0.01–0.1)** — монотонное уменьшение ошибки, сходимость к решению МНК; чем больше $\eta$ (в разумных пределах), тем меньше требуется итераций.
- **Большое $\eta$ (0.5)** — сходимость самая быстрая (в таблице — наименьшее число итераций).
- **Слишком большое $\eta$ (выше границы $2/\lambda_{max}$, в таблице расходятся 0.9 и 1.05)** — шаг «перепрыгивает» минимум, ошибка растёт, алгоритм расходится. Проверьте по таблице, где проходит граница именно для вашего датасета.

Для MSE гессиан равен $H=\frac{2}{l}X^TX$, а условие сходимости GD: $\eta < 2/\lambda_{max}(H)$. Для стандартизованных признаков $H \approx 2I$, поэтому граница около $1$, но из-за корреляции признаков $\lambda_{max}$ чуть больше, и граница оказывается ниже (значение печатается выше). Масштабирование признаков важно: без него $\lambda_{max}$ может быть огромным и пришлось бы брать очень маленькое $\eta$.

## 4. Стохастический градиентный спуск

В функцию `gradient_descent` добавлен параметр `stochastic=True`: на каждой итерации выбирается один случайный объект `i`, и градиент считается только по нему:
$$w^k = w^{k-1} - \eta \cdot 2\, x_i^T (x_i w^{k-1} - y_i).$$

Для SGD критерий $\|w^k-w^{k-1}\|<\varepsilon$ ненадёжен (на «удачном» объекте шаг может быть случайно мал), поэтому SGD запускаем на фиксированное число итераций.

In [ ]:
w_sgd, err_sgd, hist_sgd = gradient_descent(
    X_train, y_train, eta=0.01, max_iter=3000, eps=0, stochastic=True, seed=0
)
print('Веса SGD:   ', np.round(w_sgd, 3))
print('Веса МНК:   ', np.round(w_normal, 3))
print('MSE train SGD:', round(mserror(X_train, w_sgd, y_train), 3),
      '| MSE train МНК:', round(mserror(X_train, w_normal, y_train), 3))

## 5. Сравнение GD и SGD

Обучаем модель обоими методами и наносим MSE (на всей обучающей выборке) на один график.

In [ ]:
N_ITER = 1000
_, err_gd_cmp, _   = gradient_descent(X_train, y_train, eta=0.01, max_iter=N_ITER, eps=0)
_, err_sgd_cmp, _  = gradient_descent(X_train, y_train, eta=0.01, max_iter=N_ITER, eps=0,
                                      stochastic=True, seed=0)
mse_opt = mserror(X_train, w_normal, y_train)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, logx in zip(ax, [False, True]):
    a.plot(err_gd_cmp, label='GD', lw=2)
    a.plot(err_sgd_cmp, label='SGD', alpha=0.8)
    a.axhline(mse_opt, color='k', ls='--', label='MSE МНК (оптимум)')
    a.set_yscale('log'); a.set_xlabel('итерация'); a.set_ylabel('MSE')
    if logx: a.set_xscale('log')
    a.legend()
ax[0].set_title('GD и SGD, η=0.01'); ax[1].set_title('То же, логарифмическая ось итераций')
plt.tight_layout(); plt.show()

# Число итераций до достижения MSE <= 1.1 * оптимум
def iters_to(err, level):
    idx = np.where(err <= level)[0]
    return int(idx[0]) + 1 if len(idx) else None

lvl = 1.1 * mse_opt
print('Итераций до MSE <= 1.1*оптимум:  GD =', iters_to(err_gd_cmp, lvl), '| SGD =', iters_to(err_sgd_cmp, lvl))
cost_gd, cost_sgd = X_train.shape[0] * X_train.shape[1], X_train.shape[1]
print('Стоимость итерации: GD ~ O(l*n) =', cost_gd, 'операций, SGD ~ O(n) =', cost_sgd)
print('Суммарно операций до того же уровня: GD ≈', iters_to(err_gd_cmp, lvl) * cost_gd,
      '| SGD ≈', iters_to(err_sgd_cmp, lvl) * cost_sgd)

### Выводы о скорости сходимости

- **GD**: кривая ошибки гладкая и монотонно убывает, так как каждый шаг использует точный градиент по всей выборке. Но одна итерация дорогая — $O(l\cdot n)$.
- **SGD**: кривая шумная, ошибка может на отдельных итерациях расти (шаг делается по одному объекту), но в среднем падает. Итерация в $l$ раз дешевле ($O(n)$): по числу итераций методы сопоставимы, но **по затраченным вычислениям SGD достигает того же уровня ошибки на порядки быстрее** (см. вывод выше).
- **Вблизи минимума** SGD не сходится точно, а «шумит» вокруг оптимума (из-за постоянного $\eta$ и шума в выборе объекта), тогда как GD приближается к минимуму всё точнее. Для точной сходимости SGD нужно убывающее $\eta$.
- На больших выборках SGD предпочтительнее; на небольших и при требовании точного решения — GD.

## 6. Регуляризация

Реализована регуляризация обоих типов (тип выбирается по `VARIANT`: чётный — **L1**, нечётный — **L2**):

- **L2 (Ridge):** $Q(w,X) + \alpha\|w\|_2^2 \to \min$, градиент штрафа $2\alpha w$.
- **L1 (Lasso):** $Q(w,X) + \alpha\|w\|_1 \to \min$. Модуль не дифференцируем в нуле, поэтому используется проксимальный шаг градиентного спуска (мягкая пороговая обработка $w \leftarrow \mathrm{sign}(w)\max(|w|-\eta\alpha, 0)$), он даёт точные нули.

Свободный член $w_0$ не штрафуется. Для L2 кривизна функции растёт с $\alpha$, поэтому шаг подбирается как $\eta = 0.5/(1+\alpha)$ (иначе спуск расходится). Берём 20 коэффициентов регуляризации от $10^{-3}$ до $10^{3}$ (логарифмическая сетка).

In [ ]:
alphas = np.logspace(-3, 3, 20)

def fit_path(reg):
    W = []
    for a in alphas:
        # для L2 кривизна функции растёт с alpha, поэтому шаг уменьшаем: eta < 1/(1+alpha)
        eta_a = 0.5 / (1 + a) if reg == 'l2' else 0.1
        w_a, _, _ = gradient_descent(X_train, y_train, eta=eta_a, max_iter=50_000, eps=1e-12,
                                     reg=reg, alpha=a)
        W.append(w_a)
    return np.array(W)

W_main = fit_path(REG_TYPE)

fig, ax = plt.subplots(figsize=(11, 6))
for j in range(1, W_main.shape[1]):           # без w0
    ax.plot(alphas, W_main[:, j], marker='o', ms=3, label=f'w{j}')
ax.set_xscale('log')
ax.set_xlabel('коэффициент регуляризации α'); ax.set_ylabel('вес')
ax.set_title(f'Зависимость весов от α ({REG_TYPE.upper()}-регуляризация, собственный GD)')
ax.legend(ncol=2); plt.show()

Проверим результат сравнением с готовыми `Ridge` / `Lasso` из `sklearn` (у `sklearn` другая нормировка штрафа, поэтому используем пересчёт $\alpha$) и посмотрим качество на обучающей и тестовой выборках.

In [ ]:
# Для sklearn: Ridge минимизирует ||y-Xw||^2 + a||w||^2  => a_sk = alpha * l
# Lasso минимизирует (1/(2l))||y-Xw||^2 + a||w||_1       => a_sk = alpha / 2
l = X_train.shape[0]
X_tr_s, X_te_s = X_train[:, 1:], X_test[:, 1:]

W_sk = []
for a in alphas:
    m = Ridge(alpha=a * l) if REG_TYPE == 'l2' else Lasso(alpha=a / 2, max_iter=100_000)
    m.fit(X_tr_s, y_train)
    W_sk.append(m.coef_)
W_sk = np.array(W_sk)

print('Макс. отклонение весов (свой GD vs sklearn):', np.abs(W_main[:, 1:] - W_sk).max().round(4))

res = pd.DataFrame({
    'alpha': alphas,
    'train_r2': [r2_score(y_train, X_train @ w) for w in W_main],
    'test_r2':  [r2_score(y_test,  X_test  @ w) for w in W_main],
    'норма весов': [np.linalg.norm(w[1:], 1 if REG_TYPE == 'l1' else 2) for w in W_main],
    'нулевых весов': [int(np.sum(np.abs(w[1:]) < 1e-8)) for w in W_main],
})
res

In [ ]:
# Для наглядности: оба типа регуляризации рядом
W_l1, W_l2 = fit_path('l1'), fit_path('l2')

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5), sharey=False)
for a, W, name in zip(ax, [W_l1, W_l2], ['L1 (Lasso)', 'L2 (Ridge)']):
    for j in range(1, W.shape[1]):
        a.plot(alphas, W[:, j], marker='o', ms=3, label=f'w{j}')
    a.set_xscale('log'); a.set_title(name)
    a.set_xlabel('α'); a.set_ylabel('вес')
ax[0].legend(ncol=2)
plt.tight_layout(); plt.show()

### Выводы по регуляризации

- При **малых $\alpha$** решение близко к обычному МНК: веса максимальны, качество на train наилучшее.
- С **ростом $\alpha$** все веса уменьшаются по модулю, модель упрощается; при очень больших $\alpha$ все веса стремятся к нулю и модель вырождается в константу ($R^2 \to 0$).
- **L2-регуляризация** уменьшает веса плавно и одновременно, не обнуляя их точно; особенно полезна при коррелированных признаках.
- **L1-регуляризация** обнуляет веса части признаков (сначала неинформативных), т.е. выполняет отбор признаков; число нулевых весов растёт с $\alpha$.
- Оптимальное $\alpha$ лежит в области, где `test_r2` максимален — умеренная регуляризация даёт наилучший баланс между недо- и переобучением.

## Общие выводы

1. Собственная реализация GD даёт те же веса, что МНК и `sklearn`.
2. Скорость обучения критична: слишком малая — медленно, слишком большая — расходимость; масштабирование признаков упрощает подбор.
3. SGD дешевле на итерацию и быстро снижает ошибку на старте, но не сходится точно и даёт шумную кривую; GD — гладкий и точный, но дорогой.
4. Регуляризация ограничивает рост весов: L2 сжимает их, L1 зануляет и отбирает признаки.